# Purpose :

Predict the question a)

## Ideas :

- Learn to predict foreach team of the pool the result of a 1v1 duel. Then reconstruct the final result.

In [6]:
import pandas as pd
import numpy as np

teams = pd.read_csv('./data_prepared/teams.csv')

TRAIN_YEARS = list(range(1998, 2004))
TEST_YEARS  = list(range(2005, 2009))

df = teams.sort_values(['franchID', 'year']).copy()

train = df[df['year'].isin(TRAIN_YEARS)].copy()
test  = df[df['year'].isin(TEST_YEARS)].copy()

In [7]:
teams.head()

,year,lgID,tmID,franchID,confID,divID,rank,playoff,G,W,...,BenchMinor,PPG,PPC,SHA,PKG,PKC,SHF,coach_changed,conference_rank,prev_conference_rank
0,1993,NHL,ANA,ANA,WC,PC,0.500000,NaN,1.000000,0.452830,...,0.458333,0.329897,0.562334,0.571429,0.438776,0.596010,0.250000,0,0.571429,NaN
1,1994,NHL,ANA,ANA,WC,PC,0.833333,NaN,0.000000,0.132075,...,0.291667,0.010309,0.100796,0.333333,0.234694,0.109726,0.111111,0,0.785714,0.571429
2,1995,NHL,ANA,ANA,WC,PC,0.500000,NaN,0.944444,0.490566,...,0.416667,0.391753,0.694960,0.190476,0.581633,0.683292,0.277778,0,0.500000,0.785714
3,1996,NHL,ANA,ANA,WC,PC,0.166667,CSF,0.944444,0.509434,...,0.416667,0.350515,0.448276,0.476190,0.387755,0.466334,0.250000,0,0.214286,0.500000
4,1997,NHL,ANA,ANA,WC,PC,0.833333,NaN,0.944444,0.320755,...,0.166667,0.247423,0.604775,0.428571,0.489796,0.615960,0.250000,0,0.785714,0.214286


# Naive Model :

In [8]:
test['prev_conference_rank'] = test.groupby('tmID')['conference_rank'].shift(1)
test['prev_confID']    = test.groupby('tmID')['confID'].shift(1)

test.loc[test['confID'] != test['prev_confID'], 'prev_conference_rank'] = np.nan
test = test.drop(columns='prev_confID')

test['prev_conference_rank'] = test['prev_conference_rank'].astype('Float64')

In [ ]:
valid = test.dropna(subset=['prev_conference_rank', 'conference_rank']).copy()
valid['err'] = valid['prev_conference_rank'] - valid['conference_rank']

mse  = (valid['err'] ** 2).mean()
rmse = np.sqrt(mse)
print(f"MSE  (normalized): {mse:.6f}")
print(f"RMSE (normalized): {rmse:.4f}")

valid['n_teams'] = valid.groupby(['year','confID'])['conference_rank'].transform('size')
valid['err_pos'] = valid['err'] * valid['n_teams']

mae_pos  = valid['err_pos'].abs().mean()
rmse_pos = np.sqrt((valid['err_pos'] ** 2).mean())
print(f"MAE  : {mae_pos:.2f}")
print(f"RMSE : {rmse_pos:.2f}")

MSE  (normalized): 0.099748
RMSE (normalized): 0.3158
MAE  (positions): 3.65
RMSE (positions): 4.71
